# ECE 227 Project

## Topic: 1 Network Analysis and Visualization using NetworkX and Gephi

## Group Members
- Jiayi Chen | A17496530 | jic101@ucsd.edu
- Junyi Wu | A17034047 | juw040@ucsd.edu
- Matthew Alegrado | A16752818 | malegrado@ucsd.edu
- Qinpei Luo | A69035113 | qpluo@ucsd.edu
- Zihao Yang | A16751774 | ziy019@ucsd.edu

## 1. Network Preparation

We download the following models:  
1. Collaboration: [GR-QC (General Relativity and Quantum Cosmology) collaboration network](https://snap.stanford.edu/data/ca-GrQc.html). 
2. Enron email network [Enron email network](https://snap.stanford.edu/data/email-Enron.html).  
3. Social circles: Facebook [Social circles: Facebook](https://snap.stanford.edu/data/ego-Facebook.html)

In [1]:
import networkx as nx
G = nx.read_edgelist("models/facebook_combined.txt")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

Number of nodes: 4039
Number of edges: 88234


In [3]:
import re

def parse_erdos1_graph(filepath):
    """
    Parse the Erdos1 Graph text file into a NetworkX graph.

    File structure:
    - explanation lines
    - one line with number of vertices, e.g. "511"
    - then repeated blocks:
        header line: <seq> <degree> <num_outside> <author name>
        adjacency lines: one or more lines containing exactly <degree> neighbor IDs in total
    """

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.rstrip("\n") for line in f]

    # 1. find the line containing only the number of vertices
    start_idx = None
    n = None
    for i, line in enumerate(lines):
        s = line.strip()
        if re.fullmatch(r"\d+", s):
            n = int(s)
            start_idx = i + 1
            break

    if n is None:
        raise ValueError("Could not find the number-of-vertices line.")

    # 2. parse each node block
    id_to_name = {}
    neighbors_dict = {}

    idx = start_idx
    parsed_count = 0

    while parsed_count < n and idx < len(lines):
        line = lines[idx].strip()

        # skip empty lines
        if not line:
            idx += 1
            continue

        # header format:
        # seq degree something name...
        # example:
        # 1 7 16 ABBOTT, HARVEY LESLIE
        m = re.match(r"^(\d+)\s+(\d+)\s+(\d+)\s+(.+)$", line)
        if not m:
            raise ValueError(f"Header parse failed at line {idx+1}: {lines[idx]}")

        node_id = int(m.group(1))
        degree = int(m.group(2))
        # third field = number of coauthors outside this subgraph, usually not needed
        name = m.group(4).strip()

        id_to_name[node_id] = name
        idx += 1

        # 3. read adjacency numbers until we have exactly `degree` neighbors
        nbrs = []
        while len(nbrs) < degree:
            if idx >= len(lines):
                raise ValueError(f"Unexpected EOF while reading neighbors for node {node_id} ({name})")

            nbr_line = lines[idx].strip()
            idx += 1

            if not nbr_line:
                continue

            nums = [int(x) for x in nbr_line.split()]
            nbrs.extend(nums)

        if len(nbrs) > degree:
            raise ValueError(
                f"Too many neighbors read for node {node_id} ({name}). "
                f"Expected {degree}, got {len(nbrs)}"
            )

        neighbors_dict[node_id] = nbrs
        parsed_count += 1

    if parsed_count != n:
        raise ValueError(f"Parsed {parsed_count} nodes, expected {n}")

    # 4. build graph with author names as node labels
    G = nx.Graph()

    for node_id, name in id_to_name.items():
        G.add_node(node_id, name=name)

    for u_id, nbr_ids in neighbors_dict.items():
        for v_id in nbr_ids:
            G.add_edge(u_id, v_id)

    return G, id_to_name, neighbors_dict

## 4. Compare the graph to multiple random-graph models

In [4]:
import networkx as nx
import numpy as np
import random
from networkx.generators.community import LFR_benchmark_graph

def generate_advanced_models(G_real):
    
    n = G_real.number_of_nodes()
    m = G_real.number_of_edges()
    avg_degree = (2 * m) / n
    
    print(f"\n--- Generating Advanced Models for {n} nodes, {m} edges ---")

    # --- 1. Improved Stochastic Block Model (SBM) ---
    # To get non-zero clustering, we need smaller, denser blocks.
    # Aiming for ~40 nodes per community (common in social networks).
    nodes_per_comm = 40
    num_blocks = max(2, n // nodes_per_comm)
    block_sizes = [n // num_blocks] * num_blocks
    block_sizes[-1] += n - sum(block_sizes)

    # We want ~70% of edges to be internal (p_in) to create clustering.
    m_int_target = m * 0.7
    m_ext_target = m * 0.3
    
    # Calculate p_in (Internal density)
    # m_int = num_blocks * (size * (size-1) / 2) * p_in
    avg_size = n / num_blocks
    p_in = (2 * m_int_target) / (num_blocks * avg_size * (avg_size - 1))
    p_in = min(0.9, p_in) # Cap to avoid error

    # Calculate p_out (Inter-community density)
    # m_ext = (Total_pairs - Internal_pairs) * p_out
    total_pairs = n * (n - 1) / 2
    internal_pairs = num_blocks * (avg_size * (avg_size - 1) / 2)
    p_out = m_ext_target / (total_pairs - internal_pairs)
    p_out = max(1e-5, min(p_out, 1)) # Ensure small value for connectivity

    probs = np.full((num_blocks, num_blocks), p_out)
    np.fill_diagonal(probs, p_in)
    G_sbm = nx.stochastic_block_model(block_sizes, probs.tolist())

    # --- 2. LFR Benchmark Graph ---
    # This matches the power-law degree distribution AND communities.
    # If the generator fails to converge, we wrap it in a try-except.
    try:
        # Check if the number of nodes is too high for LFR to converge with these parameters
        if n > 10000:
            print(f"Too many nodes for LFR, using BA as fallback for this iteration.")
            G_lfr = nx.barabasi_albert_graph(n, int(avg_degree/2))
        else:
            G_lfr = LFR_benchmark_graph(
                n, tau1=3, tau2=1.5, mu=0.1, 
                average_degree=int(avg_degree), 
                max_degree=int(n/10), 
                min_community=int(n/20)
            )
    except Exception as e:
        print(f"LFR convergence failed, using BA as fallback for this iteration.")
        G_lfr = nx.barabasi_albert_graph(n, int(avg_degree/2))

    # --- 3. Forest Fire Model ---
    # Not in NetworkX; requires custom implementation.
    # 'p' is the burning probability (usually 0.3 - 0.4 for social nets).
    G_ff = forest_fire_impl(n, p=0.35)

    return G_sbm, G_lfr, G_ff

def forest_fire_impl(n, p):
    G = nx.Graph()
    G.add_node(0)
    for i in range(1, n):
        G.add_node(i)
        ambassador = random.randint(0, i-1)
        G.add_edge(i, ambassador)
        
        # Fire spreading logic
        current_edges = list(G.neighbors(ambassador))
        for neighbor in current_edges:
            if random.random() < p:
                G.add_edge(i, neighbor)
    return G

def analyze_graph(name):

    # Remap name to file path
    file_paths = {
        "Facebook": "models/facebook_combined.txt",
        "Email-Enron": "models/Email-Enron.txt",
    }
    # 1. Load the Graph
    # Ensure the file path is correct for your environment
    if name == "Erdos1_clean":
        G, id_to_name, neighbors_dict = parse_erdos1_graph("models/Erdos1_clean.txt")
    else:
        G = nx.read_edgelist(file_paths[name])
    n = G.number_of_nodes()
    m = G.number_of_edges()
    avg_degree = sum(dict(G.degree()).values()) / n

    print("\n====================================")
    print(f"{name}: Nodes={n}, Edges={m}")

    # 2. Generate Comparison Models
    # We use the same number of nodes (n) and similar edge density for fairness
    # Erdos-Renyi
    p = (2 * m) / (n * (n - 1)) # Probability to match edge count
    G_er = nx.fast_gnp_random_graph(n, p)

    # Watts-Strogatz (Small World)
    k = int(avg_degree)
    G_ws = nx.watts_strogatz_graph(n, k, p=0.1)

    # Barabasi-Albert (Scale-Free)
    m_ba = int(avg_degree / 2)
    G_ba = nx.barabasi_albert_graph(n, m_ba)

    G_sbm, G_lfr, G_ff = generate_advanced_models(G)

    # 3. Analysis Function
    def analyze_graph(name, G):
        print(f"\n--- Analysis for {name} ---")
        
        # Clustering: Social networks are usually high (> 0.5)
        cc = nx.average_clustering(G)
        print(f"Average Clustering Coefficient: {cc:.4f}")
        
        # Path Length: Small world networks have low path lengths
        if nx.is_connected(G):
            # If the graph is fully connected, we can compute the average shortest path length directly
            path = nx.average_shortest_path_length(G)
            print(f"Average Shortest Path Length: {path:.4f}")
        else:
            # Get the largest component if not fully connected
            gc = max(nx.connected_components(G), key=len)
            path = nx.average_shortest_path_length(G.subgraph(gc))
            print(f"Path Length (Giant Component): {path:.4f}")
        
        deep_analyze(name, G)

    def deep_analyze(name, G):
        print(f"\n--- Deep Analysis: {name} ---")
        
        # 1. Assortativity (Social networks should be > 0)
        assort = nx.degree_assortativity_coefficient(G)
        print(f"Assortativity: {assort:.4f}")
        
        # 2. Density (How sparse is it?)
        density = nx.density(G)
        print(f"Global Density: {density:.6f}")
        
        # 3. Transitivity (Global version of clustering)
        trans = nx.transitivity(G)
        print(f"Transitivity: {trans:.4f}")

        # 4. Rich-Club (Check top 1% of nodes)
        # Note: Only works for simple graphs, requires normalization for true insight
        # Returns a dict: {degree: coefficient}
        # 1. Clean the graph of self-loops
        G.remove_edges_from(nx.selfloop_edges(G))

        # 2. Now run the Rich-Club analysis
        # Note: This returns a dictionary {degree k: phi(k)}
        rc_dict = nx.rich_club_coefficient(G, normalized=False)

        # To get a single representative value, we often look at the 
        # coefficient for the highest degree (the 'richest' nodes)
        max_k = max(rc_dict.keys())
        rich_club_max = rc_dict[max_k]

        print(f"Rich-Club Coefficient (at max degree {max_k}): {rich_club_max:.4f}")

    # Run Analysis
    analyze_graph(name, G)
    # Compare with different random graph models
    analyze_graph("Erdos-Renyi (Random)", G_er)
    analyze_graph("Watts-Strogatz (Small World)", G_ws)
    analyze_graph("Barabasi-Albert (Scale-Free)", G_ba)
    analyze_graph("Stochastic Block Model (SBM)", G_sbm)
    analyze_graph("LFR Benchmark Graph", G_lfr)
    analyze_graph("Forest Fire Model", G_ff)

# Comparing different graphs
for graph_name in ["Erdos1_clean"]:
    analyze_graph(graph_name)


Erdos1_clean: Nodes=511, Edges=1604

--- Generating Advanced Models for 511 nodes, 1604 edges ---

--- Analysis for Erdos1_clean ---
Average Clustering Coefficient: 0.2633
Path Length (Giant Component): 3.8436

--- Deep Analysis: Erdos1_clean ---
Assortativity: 0.1776
Global Density: 0.012310
Transitivity: 0.2193
Rich-Club Coefficient (at max degree 43): 0.6667

--- Analysis for Erdos-Renyi (Random) ---
Average Clustering Coefficient: 0.0130
Average Shortest Path Length: 3.6697

--- Deep Analysis: Erdos-Renyi (Random) ---
Assortativity: 0.0031
Global Density: 0.011834
Transitivity: 0.0118
Rich-Club Coefficient (at max degree 13): 0.0000

--- Analysis for Watts-Strogatz (Small World) ---
Average Clustering Coefficient: 0.4501
Average Shortest Path Length: 5.5385

--- Deep Analysis: Watts-Strogatz (Small World) ---
Assortativity: -0.0021
Global Density: 0.011765
Transitivity: 0.4370
Rich-Club Coefficient (at max degree 7): 0.0182

--- Analysis for Barabasi-Albert (Scale-Free) ---
Averag

### 1. Summary of Findings

Based on the provided metrics, no single model perfectly captures every aspect of real-world networks. However, different models excel at replicating specific structural properties:

Watts-Strogatz (WS) is the best at mimicking local connectivity (Clustering).

Barabási-Albert (BA) is the best at mimicking global reach (Path Length) in large-scale networks.

Erdős-Rényi (ER) serves as a "null model," proving that real-world networks are definitively not random.

### 2. Detailed Performance Comparison

A. Clustering Coefficient (Local Structure)

The Clustering Coefficient measures how "tight-knit" a network is (i.e., if your friends are also friends with each other).
| Dataset  | Real Value | Best Model Match | Model Value |
|---|---:|---|---:|
| Facebook | 0.6055 | Watts-Strogatz | 0.5382 |
| Email    | 0.4970 | Watts-Strogatz | 0.4880 |
| CA-GrQc  | 0.5296 | Watts-Strogatz | 0.3640 |

Analysis: The WS model is the clear winner here. Real-world social and collaboration networks exhibit high "triadic closure." The ER and BA models fail significantly, often producing coefficients near zero, because they lack the mechanism to create local triangles.

B. Average Shortest Path Length (Global Efficiency)

This measures the "Degrees of Separation" required to travel from one node to another.
| Dataset | Real Value | Best Model Match | Model Value |
|---|---:|---|---:|
| Facebook | 3.6925 | Barabási-Albert | 2.5388 |
| Email | 4.0252 | Barabási-Albert | 4.0270 |
| CA-GrQc | 6.0494 | Erdős-Rényi | 5.1896 |

Analysis: For larger networks like Email, the Barabási-Albert model is remarkably accurate (4.0252 vs 4.0270). This is due to the "Scale-Free" property: the presence of massive "hubs" in both the Email data and the BA model acts as short-cuts that drastically reduce the distance between any two points in the network.

### 3. Justification of Model Suitability

The Small-World Champion: Watts-Strogatz

The WS model was specifically designed to bridge the gap between high clustering (like a lattice) and short paths (like a random graph). It accurately reflects why your social circle feels "local" yet "connected" to the rest of the world.

Verdict: Best for modeling social cohesion and community structures.

The Scale-Free Champion: Barabási-Albert

The BA model uses preferential attachment ("the rich get richer"). This explains why a few people have thousands of connections while most have very few. This inequality (hubs) is the engine behind the short path lengths seen in the Email and Facebook data.

Verdict: Best for modeling information flow, virus spreading, and network robustness.

The Baseline: Erdős-Rényi

The ER model assumes edges are formed by pure chance. Since its clustering coefficient is consistently 10x to 100x lower than real data, it proves that human interaction is driven by intent and hierarchy, not randomness.

In [5]:
G_fb = nx.read_edgelist("models/facebook_combined.txt")
G_em = nx.read_edgelist("models/Email-Enron.txt")
G_ca = nx.read_edgelist("models/CA-GrQc.txt")
nx.write_gexf(G_fb, "facebook_network.gexf")
nx.write_gexf(G_em, "email_network.gexf")
nx.write_gexf(G_ca, "ca_grqc_network.gexf")